In [4]:
import pandas as pd
import numpy as np
from rdkit.Chem import MolFromSmiles 
import os
import pickle as pkl
from pathlib import Path
path_to_neims = f'../../../data/neims'
assert Path(path_to_neims).exists()

In [16]:
from sys import version
py_ver = version.replace(' ', '').split('|')[0].replace('.', '_')
py_ver

'3_8_18'

In [17]:
nums = np.arange(0, 2206) 
exists_idx = []
not_exists_idx = []
folders = []
for idx, num in enumerate(nums):
    default = f'00000'
    folder = f''
    num_str = str(num)[::-1]
    for i, char in enumerate(default):
        try:
            folder += num_str[i]
        except:
            folder += '0'
    folder = folder[::-1]
    folders.append(folder)
folders

['00000',
 '00001',
 '00002',
 '00003',
 '00004',
 '00005',
 '00006',
 '00007',
 '00008',
 '00009',
 '00010',
 '00011',
 '00012',
 '00013',
 '00014',
 '00015',
 '00016',
 '00017',
 '00018',
 '00019',
 '00020',
 '00021',
 '00022',
 '00023',
 '00024',
 '00025',
 '00026',
 '00027',
 '00028',
 '00029',
 '00030',
 '00031',
 '00032',
 '00033',
 '00034',
 '00035',
 '00036',
 '00037',
 '00038',
 '00039',
 '00040',
 '00041',
 '00042',
 '00043',
 '00044',
 '00045',
 '00046',
 '00047',
 '00048',
 '00049',
 '00050',
 '00051',
 '00052',
 '00053',
 '00054',
 '00055',
 '00056',
 '00057',
 '00058',
 '00059',
 '00060',
 '00061',
 '00062',
 '00063',
 '00064',
 '00065',
 '00066',
 '00067',
 '00068',
 '00069',
 '00070',
 '00071',
 '00072',
 '00073',
 '00074',
 '00075',
 '00076',
 '00077',
 '00078',
 '00079',
 '00080',
 '00081',
 '00082',
 '00083',
 '00084',
 '00085',
 '00086',
 '00087',
 '00088',
 '00089',
 '00090',
 '00091',
 '00092',
 '00093',
 '00094',
 '00095',
 '00096',
 '00097',
 '00098',
 '00099',


In [18]:
smiles = []
for folder in folders:
    smi = pd.read_csv(f'canopus/{folder}/smiles.csv', header=None).values.flatten()[0]
    smiles.append(smi)

In [19]:
print(f'NUM MOLS (SMILES): {len(smiles)}')
mols = []
for smi in smiles:
    try:
        mols.append(MolFromSmiles(smi))
    except: 
        print('Issue with generating mol from smiles')

NUM MOLS (SMILES): 2206


In [20]:
def file_exists(fpath: str):
    if os.path.exists(fpath + '/annotated.sdf'):
        #print("The file exists.")
        return True
    else:
        #print("The file does not exist.")
        return False
def rdkit_3d_exists(lines: str):
    found_rdkit = False
    found_3d = False
    for line in lines:
        if 'rdkit' in line:
            found_rdkit = True
        if '3d' in line:
            found_3d = True
    return found_rdkit and found_3d
def spectrum_exists(lines: str):
    spec_num_peaks = 0
    found_spec = False
    for idx, line in enumerate(lines):
        if found_spec and line != '$$$$\n' and line != '\n':
            spec_num_peaks += 1
        if 'predicted spectrum' in line:
            found_spec = True  
    return found_spec, spec_num_peaks

In [21]:
exists_idx = []
not_exists_idx = []

for idx, folder in enumerate(folders):
    if file_exists(f'canopus/{folder}'):
        exists_idx.append(idx)
    else:
        not_exists_idx.append(idx)
print(len(exists_idx), 'exists')
print(len(not_exists_idx), 'not exists')

2206 exists
0 not exists


In [22]:
df = pd.DataFrame()
df['SMILES'] = smiles
specs = []
for idx, folder in enumerate(folders):
    if idx in not_exists_idx:
        specs.append(None)
        continue
    filename = f'canopus/{folder}/annotated.sdf'
    #print(filename)
    with open(filename, 'r') as file:
        lines = file.readlines()
        lines = [line.lower() for line in lines]
    #print(lines)
    #print(len(lines))
    #print(lines)
    spectrum_found, spec_num_peaks = spectrum_exists(lines)
    assert rdkit_3d_exists(lines), 'rdkit 3d must exist'
    assert spectrum_exists(lines), 'spectrum must exist'
    assert spec_num_peaks >= 5, 'spectrum must have more at least 5 peaks'
    spec_flag = False
    spec = [[],[]]
    for line in lines:
        if 'predicted spectrum' in line:
            spec_flag = True
            continue
        if spec_flag and line != '$$$$\n' and line != '\n':
            location, intensity = line.replace('\n', '').split()
            spec[0].append(location); spec[1].append(intensity) 
    specs.append(np.array(spec).T)
df['spec'] = specs
df.to_csv('df_neims_canopus.csv', index=False)
df.to_pickle(f"df_neims_canopus_{py_ver}.pkl")

In [23]:
df['SMILES'].to_csv('smiles_canopus.csv')

## Simulated

In [30]:
nums = np.arange(0, 10000) 
exists_idx = []
not_exists_idx = []
folders = []
for idx, num in enumerate(nums):
    default = f'00000'
    folder = f''
    num_str = str(num)[::-1]
    for i, char in enumerate(default):
        try:
            folder += num_str[i]
        except:
            folder += '0'
    folder = folder[::-1]
    folders.append(folder)

In [31]:
smiles = []
for folder in folders:
    smi = pd.read_csv(f'simulated/{folder}/smiles.csv', header=None).values.flatten()[0]
    smiles.append(smi)

In [32]:
print(f'NUM MOLS (SMILES): {len(smiles)}')
mols = []
for smi in smiles:
    try:
        mols.append(MolFromSmiles(smi))
    except: 
        print('Issue with generating mol from smiles')

NUM MOLS (SMILES): 10000


In [33]:
exists_idx = []
not_exists_idx = []

for idx, folder in enumerate(folders):
    if file_exists(f'simulated/{folder}'):
        exists_idx.append(idx)
    else:
        not_exists_idx.append(idx)
print(len(exists_idx), 'exists')
print(len(not_exists_idx), 'not exists')

9996 exists
4 not exists


In [37]:
df = pd.DataFrame()
df['SMILES'] = smiles
specs = []
for idx, folder in enumerate(folders):
    if idx in not_exists_idx:
        specs.append(None)
        continue
    filename = f'simulated/{folder}/annotated.sdf'
    #print(filename)
    with open(filename, 'r') as file:
        lines = file.readlines()
        lines = [line.lower() for line in lines]
    #print(lines)
    #print(len(lines))
    #print(lines)
    spectrum_found, spec_num_peaks = spectrum_exists(lines)
    assert rdkit_3d_exists(lines), 'rdkit 3d must exist'
    assert spectrum_exists(lines), 'spectrum must exist'
    assert spec_num_peaks >= 5, 'spectrum must have more at least 5 peaks'
    spec_flag = False
    spec = [[],[]]
    for line in lines:
        if 'predicted spectrum' in line:
            spec_flag = True
            continue
        if spec_flag and line != '$$$$\n' and line != '\n':
            location, intensity = line.replace('\n', '').split()
            spec[0].append(location); spec[1].append(intensity) 
    specs.append(np.array(spec).T)
df['spec'] = specs
df = df.dropna(subset='spec')
df.to_csv('df_neims_aug.csv', index=False)
df.to_pickle(f"df_neims_aug_{py_ver}.pkl")
df['SMILES'].to_csv('smiles_aug.csv')

In [38]:
df

,SMILES,spec
0,CCCCCCCCCCCCCCCCOCC(COP(=O)([O-])OCC[N+](C)(C)...,"[[25, 24], [26, 109], [27, 238], [28, 168], [2..."
1,CC1=C(C(=O)OC2=C1C=CC(=C2)OC(C)C(=O)NCCC(=O)O)C,"[[29, 67], [33, 58], [34, 27], [36, 32], [38, ..."
2,CC(C)C1=CC=C(C=C1)CN2CCC(C2)N(C)S(=O)(=O)C3=CC...,"[[15, 6], [27, 5], [28, 98], [30, 260], [33, 5..."
3,CCN1C2=C(C=C(C=C2)S(=O)(=O)N(C)C)N=C1CCC(=O)NC...,"[[33, 55], [34, 75], [36, 188], [37, 77], [38,..."
4,CCC(C(=O)NC1=C(C=CC(=C1)Cl)Cl)N(C2=CC=CC=C2)S(...,"[[33, 32], [34, 1], [35, 98], [36, 149], [37, ..."
...,...,...
9995,CN(C)CCN1C=C(C=N1)NC(=O)CCCC2=NC(=NO2)C3=CC=C(...,"[[33, 8], [36, 45], [37, 20], [38, 39], [39, 9..."
9996,CC(C(=O)NC(CC1=CN=CN1)C(=O)O)NC(=O)C(CCCN=C(N)...,"[[16, 72], [17, 72], [18, 74], [25, 0], [28, 5..."
9997,CCC(C(=O)OC)SC1=NN=C(S1)NC(=O)C2=NOC(=C2)C3=CC...,"[[29, 64], [33, 105], [34, 51], [35, 20], [36,..."
9998,CC1(C2CC=C(C1C2)C(=O)O)C,"[[14, 5], [16, 13], [17, 5], [26, 102], [27, 4..."
